# 5. Histopathology replication

Run on Colab with a GPU. Some cells need a fresh Colab session first (Runtime > Restart session); their first line says so.

**Data and models**
- MiniCAMELYON17 from Hugging Face (`chehablab/MiniCAMELYON17`, CC0), downloaded by CELL X1
- frozen encoders: DINOv2 ViT-B/14, Phikon (`owkin/phikon`) and Phikon-v2 (`owkin/phikon-v2`); Phikon and Phikon-v2 are under the Owkin non-commercial license

**Setup:** a standardized logistic probe (C = 1) on frozen patch embeddings, evaluated with a random patch split, folds grouped by patient, and leave one hospital out; label free harmonization (per hospital z scoring, ComBat); hospital identity predicted from the embeddings. All results are saved in `MyDrive/CARE2026_ckpts/paper_results`.

**Step 1.** CELL X1 (GPU): download the patches and embed them with DINOv2 and Phikon (Table S10).

In [ ]:
# ===================== CELL X1 (GPU): download patches and embed them with two frozen foundation models =====================
!pip install -q datasets inmoose 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, time, torch, timm
from datasets import load_dataset
from transformers import AutoModel
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"; os.makedirs(RES, exist_ok=True)
EMB, META = f"{RES}/X_camelyon_embeddings.npz", f"{RES}/X_camelyon_meta.csv"

t0 = time.time()
ds = load_dataset("chehablab/MiniCAMELYON17", split="train")
img_col = [k for k, v in ds.features.items() if type(v).__name__ == "Image"]
print(ds, "\nimage column:", img_col, f"({time.time()-t0:.0f}s)")
assert img_col, "NO-GO: this dataset has no image column"
meta = ds.remove_columns(img_col).to_pandas()
need = {"centre_id", "tumor", "patient_id"}
assert need <= set(meta.columns), f"NO-GO: missing columns {need - set(meta.columns)}"
print(pd.crosstab(meta.centre_id, meta.tumor))
print("patients per hospital:", meta.groupby("centre_id").patient_id.nunique().to_dict())
ok = [c for c, g in meta.groupby("centre_id") if g.tumor.nunique() == 2]
assert len(ok) >= 4, f"NO-GO: only {len(ok)} hospitals have both classes"
meta.to_csv(META, index=False)
print("GO: data usable\n")

if not os.path.exists(EMB):
    dino = timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224).cuda().eval()
    phik = AutoModel.from_pretrained("owkin/phikon").cuda().eval()
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda(); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()
    E1, E2, bs, t0 = [], [], 256, time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(ds), bs):
            imgs = ds[i:i + bs][img_col[0]]
            x = torch.from_numpy(np.stack([np.asarray(im.convert("RGB").resize((224, 224)), dtype=np.uint8) for im in imgs]))
            x = (x.permute(0, 3, 1, 2).float().cuda().div(255) - MEAN) / STD
            E1.append(dino(x).float().cpu().numpy())
            E2.append(phik(pixel_values=x).last_hidden_state[:, 0].float().cpu().numpy())    # CLS token
            if (i // bs) % 20 == 0: print(f"  {min(i + bs, len(ds))}/{len(ds)} ({time.time()-t0:.0f}s)")
    np.savez(EMB, dinov2=np.concatenate(E1), phikon=np.concatenate(E2))
e = np.load(EMB); print("embeddings saved:", {k: e[k].shape for k in e.files}, "| rows in meta:", len(meta))
assert all(e[k].shape[0] == len(meta) for k in e.files), "embedding count does not match metadata"

**Step 2.** CELL X2 (same session as CELL X1): the audit with two encoders.

In [ ]:
# ===================== CELL X2 (same session as X1, ~10-15 min): the audit, repeated in histopathology =====================
# (no pip here on purpose: inmoose was installed at the top of X1, before pandas was imported)
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, warnings
from inmoose.pycombat import pycombat_norm
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
warnings.filterwarnings("ignore")
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
E = np.load(f"{RES}/X_camelyon_embeddings.npz"); meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
H0 = meta.centre_id.values.astype(int)
ok = [c for c in sorted(np.unique(H0)) if meta.tumor.values[H0 == c].min() != meta.tumor.values[H0 == c].max()]
keep = np.isin(H0, ok)
y, H = meta.tumor.values.astype(int)[keep], H0[keep]
G = H * 100000 + meta.patient_id.values.astype(int)[keep]          # patient groups, unique across hospitals
FMS = {k: E[k][keep].astype(np.float32) for k in ["dinov2", "phikon"]}
print(f"{keep.sum()} patches | hospitals {ok} | patients {len(np.unique(G))} | tumor rate {y.mean():.2f}")

probe = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
def hosp_z(X):
    Z = X.copy()
    for h in ok:
        m = H == h; Z[m] = (X[m] - X[m].mean(0)) / (X[m].std(0) + 1e-6)
    return Z
def combat(X):
    sd = np.min([X[H == h].std(0) for h in ok], axis=0); k = sd > 1e-6
    return np.asarray(pycombat_norm(X[:, k].T, [str(h) for h in H])).T
rng = np.random.default_rng(0)
def cluster_ci(yt, p, g, n=500):
    ug = np.unique(g); idx = {u: np.where(g == u)[0] for u in ug}; v = []
    for _ in range(n):
        s = np.concatenate([idx[u] for u in rng.choice(ug, len(ug))])
        if yt[s].min() != yt[s].max(): v.append(roc_auc_score(yt[s], p[s]))
    return np.percentile(v, [2.5, 97.5]) if v else (np.nan, np.nan)
def oof_auc(X, splitter, groups=None):
    p = np.zeros(len(y))
    for tr, te in splitter.split(X, y, groups):
        p[te] = probe().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return roc_auc_score(y, p)

summary, per_h, dec = [], [], []
for fm, X0 in FMS.items():
    VAR = {"none": X0, "hospital-z": hosp_z(X0), "ComBat": combat(X0)}
    a_rand = oof_auc(X0, StratifiedKFold(5, shuffle=True, random_state=0))
    a_pat = oof_auc(X0, StratifiedGroupKFold(5, shuffle=True, random_state=0), G)
    loho = {}
    for vn, X in VAR.items():
        aucs = []
        for h in ok:
            tr, te = H != h, H == h
            p = probe().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
            a = roc_auc_score(y[te], p); lo, hi = cluster_ci(y[te], p, G[te]); aucs.append(a)
            per_h.append(dict(fm=fm, harmonization=vn, held_out_hospital=h, auc=a, ci=f"{lo:.3f}-{hi:.3f}"))
        loho[vn] = (np.mean(aucs), np.min(aucs))
    for vn, X in VAR.items():
        P = np.zeros((len(y), len(ok)))
        for tr, te in GroupKFold(5).split(X, H, G):
            P[te] = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000)).fit(X[tr], H[tr]).predict_proba(X[te])
        dec.append(dict(fm=fm, harmonization=vn, hospital_auc=roc_auc_score(H, P, multi_class="ovr", labels=ok),
                        hospital_acc=accuracy_score(H, np.array(ok)[P.argmax(1)])))
    summary.append(dict(fm=fm, random_split=a_rand, patient_grouped=a_pat,
                        LOHO_mean=loho["none"][0], LOHO_worst=loho["none"][1],
                        LOHO_mean_hospz=loho["hospital-z"][0], LOHO_mean_ComBat=loho["ComBat"][0]))
    print(f"  {fm} done")

S, PH, D = pd.DataFrame(summary), pd.DataFrame(per_h), pd.DataFrame(dec)
print("\n=== (1) protocol: same probe, three evaluation protocols (tumor-vs-normal AUC) ===\n" + S.round(3).to_string(index=False))
print("\n=== (2) per held-out hospital, no harmonization (patient-level cluster-bootstrap CI) ===")
w = PH[PH.harmonization == "none"].pivot(index="held_out_hospital", columns="fm", values="auc")
w["winner"] = w.idxmax(axis=1); print(w.round(3).to_string())
print(f"   overall winner: {S.set_index('fm').LOHO_mean.idxmax()} | hospitals where the other model wins: "
      f"{(w.winner != S.set_index('fm').LOHO_mean.idxmax()).sum()} of {len(w)}")
print("\n" + PH.round(3).to_string(index=False))
print("\n=== (3) can the embeddings identify the hospital? (chance accuracy = %.2f) ===" % (pd.Series(H).value_counts(normalize=True).max()))
print(D.round(3).to_string(index=False))
print("   note: after per-hospital centering, values BELOW chance are an artifact of centering plus held-out patients;"
      " read them as 'no hospital information left', not as anti-prediction")
S.to_csv(f"{RES}/X_pathology_summary.csv", index=False); PH.to_csv(f"{RES}/X_pathology_per_hospital.csv", index=False)
D.to_csv(f"{RES}/X_pathology_decodability.csv", index=False)

**Step 3.** CELL 12 (GPU): Phikon-v2 embeddings.

In [ ]:
# CELL 12 (GPU, ~5-8 min): run right after Runtime > Restart session, in the first notebook
!pip install -q datasets 2>&1 | tail -1
!pip install -q inmoose 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, time, torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModel, AutoImageProcessor
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
EMB2 = f"{RES}/X_camelyon_embeddings_phikonv2.npz"

# 1. what the dataset card says about how MiniCAMELYON17 was built
card = open(hf_hub_download("chehablab/MiniCAMELYON17", "README.md", repo_type="dataset")).read()
print("===== DATASET CARD (first 4000 characters) =====\n" + card[:4000] + "\n===== END OF CARD =====\n")

# 2. per-centre table (all centres as distributed; the analysis keeps the centres that have both classes)
meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
meta["slide"] = meta.patient_id.astype(str) + "_" + meta.node_id.astype(str)
T = meta.groupby("centre_id").agg(patients=("patient_id", "nunique"), slides=("slide", "nunique"),
                                  patches=("tumor", "size"), tumour_patches=("tumor", "sum")).reset_index()
T["tumour_pct"] = (100 * T.tumour_patches / T.patches).round(2)
T["in_analysis"] = (T.tumour_patches > 0) & (T.tumour_patches < T.patches)
print(T.to_string(index=False)); T.to_csv(f"{RES}/X_camelyon_per_centre.csv", index=False)
print("in analysis:", T[T.in_analysis][["patients", "slides", "patches", "tumour_patches"]].sum().to_dict())

# 3. Phikon-v2 embeddings for the same patches, in the same order as the October file
ds = load_dataset("chehablab/MiniCAMELYON17", split="train")
now = ds.remove_columns(["image"]).to_pandas()
same = len(now) == len(meta) and (now[["tumor", "patient_id", "centre_id", "patch_x", "patch_y"]].values ==
                                  meta[["tumor", "patient_id", "centre_id", "patch_x", "patch_y"]].values).all()
assert same, "dataset rows differ from October: STOP and check the dataset version"
if not os.path.exists(EMB2):
    proc = AutoImageProcessor.from_pretrained("owkin/phikon-v2")
    print("Phikon-v2 preprocessing:", {k: getattr(proc, k, None) for k in ["size", "crop_size", "image_mean", "image_std"]})
    model = AutoModel.from_pretrained("owkin/phikon-v2").cuda().eval()
    out, bs, t0 = [], 256, time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(ds), bs):
            x = proc(images=[im.convert("RGB") for im in ds[i:i + bs]["image"]], return_tensors="pt")["pixel_values"].cuda()
            out.append(model(pixel_values=x).last_hidden_state[:, 0].float().cpu().numpy())    # CLS token, as on the model card
            if (i // bs) % 20 == 0: print(f"  {min(i + bs, len(ds))}/{len(ds)} ({time.time()-t0:.0f}s)")
    np.savez(EMB2, phikon_v2=np.concatenate(out))
print("Phikon-v2 embeddings:", np.load(EMB2)["phikon_v2"].shape)

**Step 4.** CELL 13 (GPU, same session as CELL 12): the audit with all three encoders, nearest neighbors and hospital decodability (Table 4, Tables S11 to S13).

In [ ]:
# CELL 13 (same session as CELL 12, ~25-35 min, unattended): pathology audit with three encoders,
# nested hospital decodability, nearest-neighbour analysis
import numpy as np, pandas as pd, warnings, torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
warnings.filterwarnings("ignore")
try:
    from inmoose.pycombat import pycombat_norm; HAVE_COMBAT = True
except Exception as e:
    print("inmoose not available, ComBat rows skipped:", e); HAVE_COMBAT = False
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
E = np.load(f"{RES}/X_camelyon_embeddings.npz"); E2 = np.load(f"{RES}/X_camelyon_embeddings_phikonv2.npz")
meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
H0 = meta.centre_id.values.astype(int)
ok = [c for c in sorted(np.unique(H0)) if meta.tumor.values[H0 == c].min() != meta.tumor.values[H0 == c].max()]
keep = np.isin(H0, ok)
y, H = meta.tumor.values.astype(int)[keep], H0[keep]
G = H * 100000 + meta.patient_id.values.astype(int)[keep]          # patient groups, unique across hospitals
SL = G * 10 + meta.node_id.values.astype(int)[keep]                 # slide = patient + lymph node
FMS = {"dinov2": E["dinov2"][keep].astype(np.float32), "phikon": E["phikon"][keep].astype(np.float32),
       "phikon_v2": E2["phikon_v2"][keep].astype(np.float32)}
print(f"{keep.sum()} patches | hospitals {ok} | patients {len(np.unique(G))} | slides {len(np.unique(SL))} | tumor rate {y.mean():.2f}")

# ---------- (A) the October CELL X2 protocol, unchanged, with Phikon-v2 added as a third encoder ----------
probe = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
def hosp_z(X):
    Z = X.copy()
    for h in ok:
        m = H == h; Z[m] = (X[m] - X[m].mean(0)) / (X[m].std(0) + 1e-6)
    return Z
def combat(X):
    sd = np.min([X[H == h].std(0) for h in ok], axis=0); k = sd > 1e-6
    return np.asarray(pycombat_norm(X[:, k].T, [str(h) for h in H])).T
rng = np.random.default_rng(0)
def cluster_ci(yt, p, g, n=500):
    ug = np.unique(g); idx = {u: np.where(g == u)[0] for u in ug}; v = []
    for _ in range(n):
        s = np.concatenate([idx[u] for u in rng.choice(ug, len(ug))])
        if yt[s].min() != yt[s].max(): v.append(roc_auc_score(yt[s], p[s]))
    return np.percentile(v, [2.5, 97.5]) if v else (np.nan, np.nan)
def oof_auc(X, splitter, groups=None):
    p = np.zeros(len(y))
    for tr, te in splitter.split(X, y, groups):
        p[te] = probe().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return roc_auc_score(y, p)

summary, per_h, dec = [], [], []
for fm, X0 in FMS.items():
    VAR = {"none": X0, "hospital-z": hosp_z(X0)}
    if HAVE_COMBAT: VAR["ComBat"] = combat(X0)
    a_rand = oof_auc(X0, StratifiedKFold(5, shuffle=True, random_state=0))
    a_pat = oof_auc(X0, StratifiedGroupKFold(5, shuffle=True, random_state=0), G)
    loho = {}
    for vn, X in VAR.items():
        aucs = []
        for h in ok:
            tr, te = H != h, H == h
            p = probe().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
            a = roc_auc_score(y[te], p); lo, hi = cluster_ci(y[te], p, G[te]); aucs.append(a)
            per_h.append(dict(fm=fm, harmonization=vn, held_out_hospital=h, auc=a, ci=f"{lo:.3f}-{hi:.3f}"))
        loho[vn] = (np.mean(aucs), np.min(aucs))
    for vn, X in VAR.items():
        P = np.zeros((len(y), len(ok)))
        for tr, te in GroupKFold(5).split(X, H, G):
            P[te] = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000)).fit(X[tr], H[tr]).predict_proba(X[te])
        dec.append(dict(fm=fm, harmonization=vn, hospital_auc=roc_auc_score(H, P, multi_class="ovr", labels=ok),
                        hospital_acc=accuracy_score(H, np.array(ok)[P.argmax(1)])))
    summary.append(dict(fm=fm, random_split=a_rand, patient_grouped=a_pat,
                        LOHO_mean=loho["none"][0], LOHO_worst=loho["none"][1],
                        LOHO_mean_hospz=loho["hospital-z"][0], LOHO_mean_ComBat=loho["ComBat"][0] if HAVE_COMBAT else np.nan))
    print(f"  {fm} done")
S, PH = pd.DataFrame(summary), pd.DataFrame(per_h)
old = pd.read_csv(f"{RES}/X_pathology_summary.csv").set_index("fm"); new = S.set_index("fm")
print("\nREPRODUCTION CHECK, October/now (must be equal for dinov2 and phikon):")
for fm in ["dinov2", "phikon"]:
    print(" ", fm, " | ".join(f"{c} {old.loc[fm, c]:.4f}/{new.loc[fm, c]:.4f}" for c in old.columns))
print("\n=== (A1) same probe, three protocols (tumour AUC) ===\n" + S.round(3).to_string(index=False))
w = PH[PH.harmonization == "none"].pivot(index="held_out_hospital", columns="fm", values="auc")
print("\n=== (A2) leave-one-hospital-out, per held-out hospital, no harmonization ===\n" + w.round(3).to_string())
print("\n" + PH[PH.fm == "phikon_v2"].round(3).to_string(index=False))

# ---------- (B) hospital decodability with per-hospital z-scoring fitted on TRAINING patients only (nested) ----------
def nested_dec(X):
    P = np.zeros((len(y), len(ok)))
    for tr, te in GroupKFold(5).split(X, H, G):
        Z = np.empty_like(X)
        for h in ok:
            mt, me = tr[H[tr] == h], te[H[te] == h]
            assert len(mt) > 0, f"hospital {h} has no training patients in a fold"
            mu, sd = X[mt].mean(0), X[mt].std(0) + 1e-6
            Z[mt], Z[me] = (X[mt] - mu) / sd, (X[me] - mu) / sd
        P[te] = probe().fit(Z[tr], H[tr]).predict_proba(Z[te])
    return roc_auc_score(H, P, multi_class="ovr", labels=ok), accuracy_score(H, np.array(ok)[P.argmax(1)])
for fm, X0 in FMS.items():
    a, acc = nested_dec(X0)
    dec.append(dict(fm=fm, harmonization="hospital-z, nested (fit on training patients)", hospital_auc=a, hospital_acc=acc))
D = pd.DataFrame(dec)
print(f"\n=== (B) can the embeddings identify the hospital? (chance accuracy {pd.Series(H).value_counts(normalize=True).max():.2f}, chance AUC 0.5) ===")
print(D.round(3).to_string(index=False))

# ---------- (C) whose patch is each patch's nearest neighbour? (cosine similarity, self excluded) ----------
def chance(lbl):
    _, c = np.unique(lbl, return_counts=True); return float((c * (c - 1.0)).sum() / (len(lbl) * (len(lbl) - 1.0)))
def nn_rates(X):
    Xt = torch.nn.functional.normalize(torch.from_numpy(X).cuda().float(), dim=1).half(); J = []
    for i in range(0, len(Xt), 2048):
        sim = Xt[i:i + 2048] @ Xt.T; r_ = torch.arange(sim.shape[0], device=sim.device)
        sim[r_, r_ + i] = -2; J.append(sim.argmax(1).cpu().numpy())
    j = np.concatenate(J)
    return dict(same_slide=(SL[j] == SL).mean(), same_patient=(G[j] == G).mean(),
                same_patient_other_slide=((G[j] == G) & (SL[j] != SL)).mean(),
                same_hospital=(H[j] == H).mean(), same_label=(y[j] == y).mean())
NN = pd.DataFrame([dict(encoder=fm, **nn_rates(X0)) for fm, X0 in FMS.items()] +
                  [dict(encoder="chance", same_slide=chance(SL), same_patient=chance(G),
                        same_patient_other_slide=chance(G) - chance(SL), same_hospital=chance(H), same_label=chance(y))])
NN = NN.merge(S[["fm", "random_split", "patient_grouped"]].rename(columns={"fm": "encoder"}), on="encoder", how="left")
NN["split_gap"] = NN.random_split - NN.patient_grouped
print("\n=== (C) nearest neighbour of each patch: share from the same slide / patient / hospital / label ===")
print(NN.round(3).to_string(index=False))

S.to_csv(f"{RES}/X_pathology_summary_3fm.csv", index=False); PH.to_csv(f"{RES}/X_pathology_per_hospital_3fm.csv", index=False)
D.to_csv(f"{RES}/X_pathology_decodability_3fm.csv", index=False); NN.to_csv(f"{RES}/X_pathology_nn.csv", index=False)
print("\nsaved X_pathology_summary_3fm.csv, X_pathology_per_hospital_3fm.csv, X_pathology_decodability_3fm.csv, X_pathology_nn.csv")

**Step 5.** CELL 14: hospital decodability of a probe that sees no features, and the tumor direction across patients (Tables S12 and S14).

In [ ]:
# CELL 14 (first notebook, same session as CELL 13, ~1 min): two follow-up checks on the pathology results
#  (a) the hospital-decodability score of a model that sees NO features, under the same patient-grouped folds
#  (b) post hoc, prediction written down before running: the tumour direction is more SHARED across patients
#      for Phikon / Phikon-v2 than for DINOv2 (which would explain why only DINOv2 drops on patient-grouped splits)
import numpy as np, pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
E = np.load(f"{RES}/X_camelyon_embeddings.npz"); E2 = np.load(f"{RES}/X_camelyon_embeddings_phikonv2.npz")
meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
H0 = meta.centre_id.to_numpy(dtype=int)
ok = [c for c in sorted(np.unique(H0)) if meta.tumor.to_numpy()[H0 == c].min() != meta.tumor.to_numpy()[H0 == c].max()]
keep = np.isin(H0, ok)
y, H = meta.tumor.to_numpy(dtype=int)[keep], H0[keep]
G = H * 100000 + meta.patient_id.to_numpy(dtype=int)[keep]
SL = G * 10 + meta.node_id.to_numpy(dtype=int)[keep]
FMS = {"dinov2": E["dinov2"][keep].astype(np.float32), "phikon": E["phikon"][keep].astype(np.float32),
       "phikon_v2": E2["phikon_v2"][keep].astype(np.float32)}

# (a)
Z0 = np.zeros((len(H), 1)); P = np.zeros((len(H), len(ok)))
for tr, te in GroupKFold(5).split(Z0, H, G):
    P[te] = DummyClassifier(strategy="prior").fit(Z0[tr], H[tr]).predict_proba(Z0[te])
a0, acc0 = roc_auc_score(H, P, multi_class="ovr", labels=ok), accuracy_score(H, np.array(ok)[P.argmax(1)])
print(f"(a) NO features (intercept only), same patient-grouped folds: hospital AUC {a0:.3f}, accuracy {acc0:.3f}")
print("    compare: nested hospital-z gave AUC 0.311 / 0.309 / 0.314 and accuracy 0.286 for the three encoders")
D = pd.read_csv(f"{RES}/X_pathology_decodability_3fm.csv")
D = pd.concat([D[D.fm != "no features"], pd.DataFrame([dict(fm="no features", harmonization="intercept only (reference)",
                                                            hospital_auc=a0, hospital_acc=acc0)])], ignore_index=True)
D.to_csv(f"{RES}/X_pathology_decodability_3fm.csv", index=False)

# (b)
rows = []
for level, U in [("within slide", SL), ("within patient", G)]:
    sel = [u for u in np.unique(U) if ((U == u) & (y == 1)).sum() >= 20 and ((U == u) & (y == 0)).sum() >= 20]
    pat = {u: G[U == u][0] for u in sel}; hos = {u: H[U == u][0] for u in sel}
    print(f"\n(b) {level}: {len(sel)} units with >= 20 tumour and >= 20 normal patches | per hospital "
          f"{pd.Series([int(hos[u]) for u in sel], dtype=int).value_counts().sort_index().to_dict()}")
    if len(sel) < 3: continue
    pairs = [(i, j) for i in range(len(sel)) for j in range(len(sel)) if i != j and pat[sel[i]] != pat[sel[j]]]
    for fm, X in FMS.items():
        Zs = (X - X.mean(0)) / (X.std(0) + 1e-6)                    # same standardisation as the probe
        d = np.array([Zs[(U == u) & (y == 1)].mean(0) - Zs[(U == u) & (y == 0)].mean(0) for u in sel])
        un = d / np.linalg.norm(d, axis=1, keepdims=True)
        cos = [float(un[i] @ un[j]) for i, j in pairs if i < j]
        auc = [roc_auc_score(y[U == sel[j]], Zs[U == sel[j]] @ d[i]) for i, j in pairs]
        rows.append(dict(level=level, encoder=fm, units=len(sel), pairs=len(cos),
                         mean_cosine=np.mean(cos), min_cosine=np.min(cos),
                         auc_direction_from_another_patient=np.mean(auc), worst_auc=np.min(auc)))
R = pd.DataFrame(rows); print("\n" + R.round(3).to_string(index=False))
R.to_csv(f"{RES}/X_pathology_tumour_direction.csv", index=False)
print("\nsaved X_pathology_tumour_direction.csv; decodability table now includes the no-feature reference")

**Step 6.** CELL 16: intervals for the tumor direction AUC (Table S14).

In [ ]:
# CELL 16 (first notebook, ~1 min): 95% CI for the cross-patient tumour-direction AUC (bootstrap over patients)
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
E = np.load(f"{RES}/X_camelyon_embeddings.npz"); E2 = np.load(f"{RES}/X_camelyon_embeddings_phikonv2.npz")
meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
H0 = meta.centre_id.to_numpy(dtype=int)
ok = [c for c in sorted(np.unique(H0)) if meta.tumor.to_numpy()[H0 == c].min() != meta.tumor.to_numpy()[H0 == c].max()]
keep = np.isin(H0, ok); y, H = meta.tumor.to_numpy(dtype=int)[keep], H0[keep]
G = H * 100000 + meta.patient_id.to_numpy(dtype=int)[keep]; SL = G * 10 + meta.node_id.to_numpy(dtype=int)[keep]
FMS = {"dinov2": E["dinov2"][keep].astype(np.float32), "phikon": E["phikon"][keep].astype(np.float32),
       "phikon_v2": E2["phikon_v2"][keep].astype(np.float32)}
sel = [u for u in np.unique(SL) if ((SL == u) & (y == 1)).sum() >= 20 and ((SL == u) & (y == 0)).sum() >= 20]
pat = np.array([G[SL == u][0] for u in sel]); n = len(sel)
A = {}
for fm, X in FMS.items():
    Zs = (X - X.mean(0)) / (X.std(0) + 1e-6)
    d = np.array([Zs[(SL == u) & (y == 1)].mean(0) - Zs[(SL == u) & (y == 0)].mean(0) for u in sel])
    M = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            if pat[i] != pat[j]: M[i, j] = roc_auc_score(y[SL == sel[j]], Zs[SL == sel[j]] @ d[i])
    A[fm] = M
def mean_pairs(M, idx):
    sub = M[np.ix_(idx, idx)]; return np.nanmean(sub[~np.eye(len(idx), dtype=bool)])
rng = np.random.default_rng(0); B = 2000; boot = {fm: [] for fm in A}
for _ in range(B):
    idx = rng.integers(0, n, n)
    for fm in A: boot[fm].append(mean_pairs(A[fm], idx))
boot = {fm: np.array(v) for fm, v in boot.items()}
rows = []
for fm in A:
    lo, hi = np.nanpercentile(boot[fm], [2.5, 97.5])
    rows.append(dict(quantity=f"{fm}: mean AUC, direction from another patient", estimate=mean_pairs(A[fm], np.arange(n)), ci=f"{lo:.3f}-{hi:.3f}", p=np.nan))
for a in ["phikon", "phikon_v2"]:
    dd = boot[a] - boot["dinov2"]; dd = dd[np.isfinite(dd)]; lo, hi = np.percentile(dd, [2.5, 97.5])
    rows.append(dict(quantity=f"{a} minus dinov2", estimate=mean_pairs(A[a], np.arange(n)) - mean_pairs(A["dinov2"], np.arange(n)),
                     ci=f"{lo:+.3f},{hi:+.3f}", p=max(1 / len(dd), min(1.0, 2 * min((dd <= 0).mean(), (dd >= 0).mean())))))
R = pd.DataFrame(rows); print(f"{n} slides from {len(np.unique(pat))} patients, bootstrap over slides, {B} draws\n" + R.round(3).to_string(index=False))
R.to_csv(f"{RES}/X_pathology_tumour_direction_ci.csv", index=False)

**Step 7.** CELL 17 (GPU): embeddings of native resolution central crops.

In [ ]:
# CELL 17 (GPU, ~5 min): run after Runtime > Restart session. Post hoc sensitivity, design fixed before running:
# same 26,753 patches, but a CENTRAL 224x224 crop at native 0.5 um/px (20x) instead of resizing 512 -> 224,
# so the input matches the magnification the pathology encoders were trained at. Same three frozen encoders,
# same normalisation (ImageNet mean/std), CLS token for Phikon / Phikon-v2, as in October.
!pip install -q datasets 2>&1 | tail -1
!pip install -q inmoose 2>&1 | tail -1
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, pandas as pd, os, time, torch, timm
from datasets import load_dataset
from transformers import AutoModel
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
OUTF = f"{RES}/X_camelyon_embeddings_native224.npz"
meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
ds = load_dataset("chehablab/MiniCAMELYON17", split="train")
now = ds.remove_columns(["image"]).to_pandas()
assert len(now) == len(meta) and (now[["tumor", "patient_id", "centre_id", "patch_x", "patch_y"]].to_numpy() ==
                                  meta[["tumor", "patient_id", "centre_id", "patch_x", "patch_y"]].to_numpy()).all(), "rows differ: STOP"
print("patch size:", ds[0]["image"].size)

def center224(im):
    im = im.convert("RGB"); w, h = im.size; l, t = (w - 224) // 2, (h - 224) // 2
    return np.asarray(im.crop((l, t, l + 224, t + 224)), dtype=np.uint8)

if not os.path.exists(OUTF):
    enc = {"dinov2": timm.create_model("vit_base_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224).cuda().eval(),
           "phikon": AutoModel.from_pretrained("owkin/phikon").cuda().eval(),
           "phikon_v2": AutoModel.from_pretrained("owkin/phikon-v2").cuda().eval()}
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda(); STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()
    out = {k: [] for k in enc}; bs, t0 = 256, time.time()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for i in range(0, len(ds), bs):
            x = torch.from_numpy(np.stack([center224(im) for im in ds[i:i + bs]["image"]]))
            x = (x.permute(0, 3, 1, 2).float().cuda().div(255) - MEAN) / STD
            out["dinov2"].append(enc["dinov2"](x).float().cpu().numpy())
            out["phikon"].append(enc["phikon"](pixel_values=x).last_hidden_state[:, 0].float().cpu().numpy())
            out["phikon_v2"].append(enc["phikon_v2"](pixel_values=x).last_hidden_state[:, 0].float().cpu().numpy())
            if (i // bs) % 20 == 0: print(f"  {min(i + bs, len(ds))}/{len(ds)} ({time.time()-t0:.0f}s)")
    np.savez(OUTF, **{k: np.concatenate(v) for k, v in out.items()})
e = np.load(OUTF); print("saved:", {k: e[k].shape for k in e.files})

**Step 8.** CELL 18 (GPU, same session as CELL 17): the audit on native resolution crops (Tables S11, S14 and S15).

In [ ]:
# CELL 18 (same session as CELL 17, ~30-35 min unattended): full pathology evaluation on the native-resolution
# embeddings (same code as CELLS 13, 14, 16), printed next to the resized results used so far
import numpy as np, pandas as pd, warnings, torch
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import roc_auc_score, accuracy_score
warnings.filterwarnings("ignore")
try:
    from inmoose.pycombat import pycombat_norm; HAVE_COMBAT = True
except Exception as e:
    print("inmoose not available, ComBat rows skipped:", e); HAVE_COMBAT = False
RES = "/content/drive/MyDrive/CARE2026_ckpts/paper_results"
E = np.load(f"{RES}/X_camelyon_embeddings_native224.npz"); meta = pd.read_csv(f"{RES}/X_camelyon_meta.csv")
H0 = meta.centre_id.to_numpy(dtype=int)
ok = [c for c in sorted(np.unique(H0)) if meta.tumor.to_numpy()[H0 == c].min() != meta.tumor.to_numpy()[H0 == c].max()]
keep = np.isin(H0, ok)
y, H = meta.tumor.to_numpy(dtype=int)[keep], H0[keep]
G = H * 100000 + meta.patient_id.to_numpy(dtype=int)[keep]
SL = G * 10 + meta.node_id.to_numpy(dtype=int)[keep]
FMS = {k: E[k][keep].astype(np.float32) for k in ["dinov2", "phikon", "phikon_v2"]}
print(f"{keep.sum()} patches | hospitals {ok} | patients {len(np.unique(G))}")

# ---------- (A) CELL X2 protocol, as in CELL 13 ----------
probe = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
def hosp_z(X):
    Z = X.copy()
    for h in ok:
        m = H == h; Z[m] = (X[m] - X[m].mean(0)) / (X[m].std(0) + 1e-6)
    return Z
def combat(X):
    sd = np.min([X[H == h].std(0) for h in ok], axis=0); k = sd > 1e-6
    return np.asarray(pycombat_norm(X[:, k].T, [str(h) for h in H])).T
rng = np.random.default_rng(0)
def cluster_ci(yt, p, g, n=500):
    ug = np.unique(g); idx = {u: np.where(g == u)[0] for u in ug}; v = []
    for _ in range(n):
        s = np.concatenate([idx[u] for u in rng.choice(ug, len(ug))])
        if yt[s].min() != yt[s].max(): v.append(roc_auc_score(yt[s], p[s]))
    return np.percentile(v, [2.5, 97.5]) if v else (np.nan, np.nan)
def oof_auc(X, splitter, groups=None):
    p = np.zeros(len(y))
    for tr, te in splitter.split(X, y, groups):
        p[te] = probe().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return roc_auc_score(y, p)
summary, per_h, dec = [], [], []
for fm, X0 in FMS.items():
    VAR = {"none": X0, "hospital-z": hosp_z(X0)}
    if HAVE_COMBAT: VAR["ComBat"] = combat(X0)
    a_rand = oof_auc(X0, StratifiedKFold(5, shuffle=True, random_state=0))
    a_pat = oof_auc(X0, StratifiedGroupKFold(5, shuffle=True, random_state=0), G)
    loho = {}
    for vn, X in VAR.items():
        aucs = []
        for h in ok:
            tr, te = H != h, H == h
            p = probe().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
            a = roc_auc_score(y[te], p); lo, hi = cluster_ci(y[te], p, G[te]); aucs.append(a)
            per_h.append(dict(fm=fm, harmonization=vn, held_out_hospital=h, auc=a, ci=f"{lo:.3f}-{hi:.3f}"))
        loho[vn] = (np.mean(aucs), np.min(aucs))
    for vn, X in VAR.items():
        P = np.zeros((len(y), len(ok)))
        for tr, te in GroupKFold(5).split(X, H, G):
            P[te] = probe().fit(X[tr], H[tr]).predict_proba(X[te])
        dec.append(dict(fm=fm, harmonization=vn, hospital_auc=roc_auc_score(H, P, multi_class="ovr", labels=ok),
                        hospital_acc=accuracy_score(H, np.array(ok)[P.argmax(1)])))
    summary.append(dict(fm=fm, random_split=a_rand, patient_grouped=a_pat, LOHO_mean=loho["none"][0], LOHO_worst=loho["none"][1],
                        LOHO_mean_hospz=loho["hospital-z"][0], LOHO_mean_ComBat=loho["ComBat"][0] if HAVE_COMBAT else np.nan))
    print(f"  {fm} done")
S, PH = pd.DataFrame(summary), pd.DataFrame(per_h)

# ---------- (B) nested hospital decodability, as in CELL 13 ----------
def nested_dec(X):
    P = np.zeros((len(y), len(ok)))
    for tr, te in GroupKFold(5).split(X, H, G):
        Z = np.empty_like(X)
        for h in ok:
            mt, me = tr[H[tr] == h], te[H[te] == h]
            mu, sd = X[mt].mean(0), X[mt].std(0) + 1e-6
            Z[mt], Z[me] = (X[mt] - mu) / sd, (X[me] - mu) / sd
        P[te] = probe().fit(Z[tr], H[tr]).predict_proba(Z[te])
    return roc_auc_score(H, P, multi_class="ovr", labels=ok), accuracy_score(H, np.array(ok)[P.argmax(1)])
for fm, X0 in FMS.items():
    a, acc = nested_dec(X0)
    dec.append(dict(fm=fm, harmonization="hospital-z, nested (fit on training patients)", hospital_auc=a, hospital_acc=acc))
D = pd.DataFrame(dec)

# ---------- (C) nearest neighbours, as in CELL 13 ----------
def nn_rates(X):
    Xt = torch.nn.functional.normalize(torch.from_numpy(X).cuda().float(), dim=1).half(); J = []
    for i in range(0, len(Xt), 2048):
        sim = Xt[i:i + 2048] @ Xt.T; r_ = torch.arange(sim.shape[0], device=sim.device)
        sim[r_, r_ + i] = -2; J.append(sim.argmax(1).cpu().numpy())
    j = np.concatenate(J)
    return dict(same_slide=(SL[j] == SL).mean(), same_patient=(G[j] == G).mean(),
                same_hospital=(H[j] == H).mean(), same_label=(y[j] == y).mean())
NN = pd.DataFrame([dict(encoder=fm, **nn_rates(X0)) for fm, X0 in FMS.items()])

# ---------- (D) tumour direction from another patient, with bootstrap CI, as in CELL 16 ----------
sel = [u for u in np.unique(SL) if ((SL == u) & (y == 1)).sum() >= 20 and ((SL == u) & (y == 0)).sum() >= 20]
pat = np.array([G[SL == u][0] for u in sel]); n = len(sel); AM = {}
for fm, X in FMS.items():
    Zs = (X - X.mean(0)) / (X.std(0) + 1e-6)
    d = np.array([Zs[(SL == u) & (y == 1)].mean(0) - Zs[(SL == u) & (y == 0)].mean(0) for u in sel])
    M = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(n):
            if pat[i] != pat[j]: M[i, j] = roc_auc_score(y[SL == sel[j]], Zs[SL == sel[j]] @ d[i])
    AM[fm] = M
def mean_pairs(M, idx):
    sub = M[np.ix_(idx, idx)]; return np.nanmean(sub[~np.eye(len(idx), dtype=bool)])
rb = np.random.default_rng(0); BT = {fm: [] for fm in AM}
for _ in range(2000):
    idx = rb.integers(0, n, n)
    for fm in AM: BT[fm].append(mean_pairs(AM[fm], idx))
TD = []
for fm in AM:
    lo, hi = np.nanpercentile(np.array(BT[fm]), [2.5, 97.5])
    TD.append(dict(encoder=fm, auc_direction_from_another_patient=mean_pairs(AM[fm], np.arange(n)), ci=f"{lo:.3f}-{hi:.3f}"))
for a in ["phikon", "phikon_v2"]:
    dd = np.array(BT[a]) - np.array(BT["dinov2"]); dd = dd[np.isfinite(dd)]; lo, hi = np.percentile(dd, [2.5, 97.5])
    TD.append(dict(encoder=f"{a} minus dinov2", ci=f"{lo:+.3f},{hi:+.3f}",
                   auc_direction_from_another_patient=mean_pairs(AM[a], np.arange(n)) - mean_pairs(AM["dinov2"], np.arange(n))))
TD = pd.DataFrame(TD)

# ---------- side by side with the resized results ----------
OS = pd.read_csv(f"{RES}/X_pathology_summary_3fm.csv").set_index("fm"); NS = S.set_index("fm")
rows = [dict(encoder=fm, metric=c, resized=OS.loc[fm, c], native=NS.loc[fm, c], change=NS.loc[fm, c] - OS.loc[fm, c])
        for fm in FMS for c in ["random_split", "patient_grouped", "LOHO_mean", "LOHO_worst", "LOHO_mean_hospz", "LOHO_mean_ComBat"]]
print("\n=== (A) tumour AUC, resized (as submitted) vs native-resolution crop ===\n" + pd.DataFrame(rows).round(3).to_string(index=False))
w = PH[PH.harmonization == "none"].pivot(index="held_out_hospital", columns="fm", values="auc")
print("\nnative, leave-one-hospital-out per hospital (no harmonization):\n" + w.round(3).to_string())
OD = pd.read_csv(f"{RES}/X_pathology_decodability_3fm.csv")
print("\n=== (B) hospital decodability: native (resized in brackets) ===")
for fm in FMS:
    for hz in ["none", "hospital-z, nested (fit on training patients)"]:
        a_ = D[(D.fm == fm) & (D.harmonization == hz)].iloc[0]; b_ = OD[(OD.fm == fm) & (OD.harmonization == hz)].iloc[0]
        print(f"  {fm:9s} {hz[:22]:22s} AUC {a_.hospital_auc:.3f} ({b_.hospital_auc:.3f})  acc {a_.hospital_acc:.3f} ({b_.hospital_acc:.3f})")
ONN = pd.read_csv(f"{RES}/X_pathology_nn.csv").set_index("encoder")
print("\n=== (C) nearest neighbour: native (resized in brackets) ===")
for _, r in NN.iterrows():
    print(f"  {r.encoder:9s} same slide {r.same_slide:.3f} ({ONN.loc[r.encoder, 'same_slide']:.3f})  "
          f"same label {r.same_label:.3f} ({ONN.loc[r.encoder, 'same_label']:.3f})")
print(f"\n=== (D) tumour direction from another patient, native ({n} slides) ===\n" + TD.round(3).to_string(index=False))
print("resized, for comparison:\n" + pd.read_csv(f"{RES}/X_pathology_tumour_direction_ci.csv").round(3).to_string(index=False))

S.to_csv(f"{RES}/X_pathology_summary_native224.csv", index=False); PH.to_csv(f"{RES}/X_pathology_per_hospital_native224.csv", index=False)
D.to_csv(f"{RES}/X_pathology_decodability_native224.csv", index=False); NN.to_csv(f"{RES}/X_pathology_nn_native224.csv", index=False)
TD.to_csv(f"{RES}/X_pathology_tumour_direction_native224.csv", index=False)
print("\nsaved all *_native224.csv files")